## Basic RAG Pipeline Example

In [ ]:
# !pip install -U transformers sentence-transformers faiss-cpu ipywidgets

### RAW Knowledge Base

In [ ]:
DOCUMENTS = [
    "The Eiffel Tower is located in Paris.",
    "The Great Wall of China is visible from space.",
    "The Amazon rainforest produces 20% of the world's oxygen.",
    "A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron.",
    "Bozanium is a rare earth element found mainly in the country of Bozania.",
    "Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.",
]

In [ ]:
query = "What can you tell me about Bozanium?"

## Initialize HuggingFace Models

We require two models here, one for embedding and another for the LLM itself. For each we require the tokenizer which goes with it.

In [ ]:
HF_LLM_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

HF_EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"   # "all-mpnet-base-v2"

DEVICE = "cuda"   # "cpu"

In [ ]:
import os

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

In [ ]:
LLM_TOKENIZER = AutoTokenizer.from_pretrained(HF_LLM_MODEL_ID)
LLM_MODEL = AutoModelForCausalLM.from_pretrained(HF_LLM_MODEL_ID, dtype=torch.bfloat16, device_map=DEVICE)

EMBEDDING_MODEL = SentenceTransformer(model_name_or_path=HF_EMBEDDING_MODEL_ID, device=DEVICE).to(DEVICE)

## Embedding

In [ ]:
normalized_doc_embeddings = EMBEDDING_MODEL.encode(DOCUMENTS)

In [ ]:
normalized_doc_embeddings.shape

## Indexing

We'll use an in-memory vector database from Meta - *FAISS* (what does it sound to you like 😊?)

In [ ]:
import faiss

The `sentence_transformers` package takes care of pooling and normalization for us.

In [ ]:
# https://www.restack.io/p/vector-database-answer-faiss-cosine-similarity-cat-ai
# IndexFlatIP - Inner Product - equivalent to cosine-similarity, given the embeddings are normalized
FAISS_INDEX = faiss.IndexFlatIP(normalized_doc_embeddings.shape[-1])

In [ ]:
FAISS_INDEX.add(normalized_doc_embeddings)

In [ ]:
# Persisting. Note that the documents are not persisted alongside the embeddings.

# faiss.write_index(FAISS_INDEX, "doc-embeddings.index")
# FAISS_INDEX = faiss.read_index(FAISS_INDEX,"doc-embeddings.index")

## Retriever

Returns the most relevant documents based on the similarity of their embeddings to the query embedding.

In [ ]:
query_embedding = EMBEDDING_MODEL.encode([query])

In [ ]:
query_embedding.shape

In [ ]:
faiss_similarity_scores, faiss_indices = FAISS_INDEX.search(query_embedding, 5)

In [ ]:
faiss_similarity_scores

In [ ]:
faiss_indices

In [ ]:
print("Query:", query)
[f"[{score:.3f}] {DOCUMENTS[i]}" for i, score in zip(faiss_indices[0], faiss_similarity_scores[0])]

In [ ]:
def retrieve_documents(query_text, top_k=3, similarity_cutoff=0.4):
    
    query_embedding = EMBEDDING_MODEL.encode([query_text])
    faiss_similarity_scores, faiss_indices = FAISS_INDEX.search(query_embedding, top_k)
    
    return [DOCUMENTS[i] for rag_doc_i, i in enumerate(faiss_indices[0]) if faiss_similarity_scores[0][rag_doc_i] >= similarity_cutoff]

In [ ]:
retrieve_documents(query)

## Generator

In [ ]:
# pipeline?

In [ ]:
GENERATOR = pipeline(task="text-generation", model=LLM_MODEL, tokenizer=LLM_TOKENIZER)

**Without** contextual augmentation:

In [ ]:
response = GENERATOR(query, max_new_tokens=300, return_full_text=False, temperature=0.01)
print(response[0]["generated_text"])

Well, its all hallucinations...

In [ ]:
def generate_answer(query_text, rag_documents: list[str]) -> str:

    context = f"""
    Context information is provided to augment your knowledge. 
    Context: {os.linesep.join(rag_documents)}
    """ if rag_documents else ""

    prompt = f"""
    You are a smart AI assistant. Only reply with your final answer.
    {context}\n
    Question: {query_text}\n
    Answer:"""
    
    response = GENERATOR(prompt, max_new_tokens=300, return_full_text=False, temperature=0.01)

    return response[0]["generated_text"]

### RAG Pipeline

In [ ]:
from IPython.display import display, Markdown

In [ ]:
def rag_pipeline(query_text, use_context=True, display_answer=True):

    rag_documents = retrieve_documents(query_text) if use_context else None
    answer = generate_answer(query_text, rag_documents)

    if display_answer:
        print("Query:", query_text, f"" if use_context else "// NO CONTEXT")
        display(Markdown(answer))
    else:
        return answer

## Testing

In [ ]:
query_1 = "What can you tell me about Bozanium?"

In [ ]:
rag_pipeline(query_1, use_context=False)

In [ ]:
retrieve_documents(query_1)

In [ ]:
rag_pipeline(query_1, use_context=True)

In [ ]:
query_2 = "What could help make firemen suits made of plastic more fireproof?"

In [ ]:
rag_pipeline(query_2, use_context=False)

In [ ]:
retrieve_documents(query_2)

In [ ]:
rag_pipeline(query_2, use_context=True)

In [ ]:
query_3 = "Have new elements been discovered lately?"

In [ ]:
rag_pipeline(query_3, use_context=False)

In [ ]:
retrieve_documents(query_3)

In [ ]:
rag_pipeline(query_3, use_context=True)

In [ ]:
query_4 = "What is the importance of rainforests?"

In [ ]:
rag_pipeline(query_4, use_context=False)

In [ ]:
retrieve_documents(query_4)

In [ ]:
rag_pipeline(query_4, use_context=True)

<hr/>